In [6]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [7]:
df = pd.read_csv("../data/raw/TTC Subway Delay Data since 2025.csv")
df.head()

,_id,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle
0,1,2025-01-01,02:10,Wednesday,BATHURST STATION,MUSAN,5,9,E,BD,5227
1,2,2025-01-01,02:30,Wednesday,DUNDAS STATION,MUIRS,0,0,NaN,YU,0
2,3,2025-01-01,02:32,Wednesday,BROADVIEW STATION,PUMST,0,0,E,BD,0
3,4,2025-01-01,02:58,Wednesday,KEELE STATION,EUSC,0,0,W,BD,5293
4,5,2025-01-01,02:58,Wednesday,COXWELL STATION,SUAE,0,0,NaN,BD,0


In [8]:
print(df.shape)
df.info()
df.isna().sum()

(45475, 11)
<class 'pandas.DataFrame'>
RangeIndex: 45475 entries, 0 to 45474
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   _id        45475 non-null  int64
 1   Date       45475 non-null  str  
 2   Time       45475 non-null  str  
 3   Day        45475 non-null  str  
 4   Station    45475 non-null  str  
 5   Code       45475 non-null  str  
 6   Min Delay  45475 non-null  int64
 7   Min Gap    45475 non-null  int64
 8   Bound      29615 non-null  str  
 9   Line       45298 non-null  str  
 10  Vehicle    45475 non-null  int64
dtypes: int64(4), str(7)
memory usage: 3.8 MB


_id              0
Date             0
Time             0
Day              0
Station          0
Code             0
Min Delay        0
Min Gap          0
Bound        15860
Line           177
Vehicle          0
dtype: int64

In [12]:
# Distribution of delay minutes
print(df["Min Delay"].describe())
print()
print("Share of 0-min delays:", (df["Min Delay"] == 0).mean().round(3))
print("Share of delays >= 5 min:", (df["Min Delay"] >= 5).mean().round(3))
print("Share of delays >= 10 min:", (df["Min Delay"] >= 10).mean().round(3))

count    45475.000000
mean         2.684354
std         12.669995
min          0.000000
25%          0.000000
50%          0.000000
75%          3.000000
max        900.000000
Name: Min Delay, dtype: float64

Share of 0-min delays: 0.652
Share of delays >= 5 min: 0.18
Share of delays >= 10 min: 0.058


In [10]:
# Incident counts by line
df["Line"].value_counts()

Line
YU                        23134
BD                        19758
SHP                        1772
YU/BD                       470
YUS/BD                       87
YUS                          13
YU/BD/SHP                    11
BD/YU                         7
BD/YUS                        5
SRT                           5
YU/ BD                        4
YUS/ BD                       3
YU / BD                       2
EC                            2
YUS / BD                      2
YU/BD LINES                   2
LINE 1                        1
95 YORK MILLS                 1
508 LAKESHORE                 1
116 MORNINGSIDE               1
YUS/ BD/ SHP                  1
62 MORTIMER                   1
LINE 2 - BLOOR DANFORT        1
YU -BD LINES                  1
29 DUFFERIN                   1
60 STEELES WEST               1
76 ROYAL YORK SOUTH           1
50 BURNHAMTHORPE              1
BD/YUS/SHP/FWLRT/ECLRT        1
506 CARLTON                   1
WOODBINE STN                  1
999

In [11]:
# Number of unique station names and the most frequent ones
print("Unique stations:", df["Station"].nunique())
df["Station"].value_counts().head(30)

Unique stations: 1251


Station
BLOOR STATION             1849
FINCH STATION             1582
KENNEDY BD STATION        1577
KIPLING STATION           1508
YONGE BD STATION          1109
EGLINTON STATION          1093
VMC STATION                944
UNION STATION              937
WILSON STATION             918
WARDEN STATION             835
VICTORIA PARK STATION      798
WELLESLEY STATION          785
BROADVIEW STATION          708
COXWELL STATION            662
ST CLAIR STATION           647
ST GEORGE YUS STATION      639
KEELE STATION              637
DAVISVILLE STATION         636
ST GEORGE BD STATION       631
DUNDAS WEST STATION        614
MAIN STREET STATION        612
COLLEGE STATION            610
SHEPPARD WEST STATION      605
ISLINGTON STATION          604
BATHURST STATION           559
ST CLAIR WEST STATION      537
SPADINA BD STATION         537
OSSINGTON STATION          537
QUEEN STATION              525
SHEPPARD-YONGE STATION     525
Name: count, dtype: int64

## Key findings so far
- 65% of incidents have a 0-minute delay; 18% are ≥ 5 min and 5.8% are ≥ 10 min.
- Target: classify delays as long if ≥ 5 min (better class balance than 10 min, and matches the common 5-minute threshold for reporting subway outages).
- `Line` is messy: multi-line combos written many ways, the old `YUS` code, and a few bus routes recorded by mistake. Needs cleaning.
- Max delay is 900 min, likely an outlier to inspect.

In [13]:
# How much of the data do the most frequent station names cover?
counts = df["Station"].value_counts()
for n in [50, 75, 100, 150]:
    coverage = counts.head(n).sum() / len(df)
    print(f"Top {n} names cover {coverage:.1%} of incidents")

Top 50 names cover 74.4% of incidents
Top 75 names cover 90.6% of incidents
Top 100 names cover 94.2% of incidents
Top 150 names cover 96.1% of incidents


In [14]:
# Look at the long tail: names outside the top 100
pd.set_option("display.max_rows", 100)
counts.iloc[100:160]

Station
FINCH STATION - PLATFO    32
KIPLING STATION PLATFO    32
YONGE UNIVERSITY SUBWA    31
VMC STATION (PLATFORM     30
BLOOR DANFORTH LINE 2     26
YONGE UNIVERSITY AND B    25
KEELE YARD                25
UNION STATION (TOWARDS    24
GREENWOOD CARHOUSE        24
LINE 2 - BLOOR DANFORT    23
VMC STATION PLATFORM 2    22
NORTH YORK CENTRE STAT    21
LINE 2 - BLOOR-DANFORT    20
LINE 1- YONGE UNIVERSI    20
BLOOR-DANFORTH LINE 2     19
SPADINA YU STATION        19
LINE 1 YONGE-UNIVERSIT    18
VMC STATION PLATFORM 1    18
GREENWOOD WYE             17
VMC STATION - PLATFORM    17
KIPLING STATION (APPRO    16
ST. GEORGE STATION        16
YONGE-UNIVERSITY LINE     16
KENNEDY BD STATION (AP    15
ISLINGTON STATION (APP    15
UNION TO KING STATIONS    15
LINE 1 - YONGE UNIVERS    14
ST GEORGE STATION         14
BLOOR DANFORTH LINE       14
LINE 2 BLOOR-DANFORTH     14
WILSON CARHOUSE           14
YONGE/UNIVERSITY AND B    13
DAVISVILLE CARHOUSE       13
YONGE STATION             13
UNION 

In [15]:
import re

def clean_station(name):
    """Map a raw station name to a cleaned category."""
    if pd.isna(name):
        return "UNKNOWN"
    s = str(name).upper().replace(".", "")
    s = re.sub(r"\s+", " ", s).strip()

    # Station names with extra suffixes, e.g. "FINCH STATION - PLATFO"
    if "STATION" in s:
        return s.split("STATION")[0].strip() + " STATION"
    # Yards, carhouses and track infrastructure
    if any(word in s for word in ["YARD", "CARHOUSE", "WYE"]):
        return "YARD"
    # Incidents affecting a whole line rather than one station
    if "LINE" in s or "SUBWA" in s:
        return "LINE-WIDE"
    return "OTHER"

df["station_clean"] = df["Station"].apply(clean_station)
print("Unique stations after cleaning:", df["station_clean"].nunique())

Unique stations after cleaning: 126


In [16]:
# Check the result: most common cleaned names, and how much falls into each catch-all group
print(df["station_clean"].value_counts().head(10))
print()
for group in ["YARD", "LINE-WIDE", "OTHER", "UNKNOWN"]:
    print(f"{group}: {(df['station_clean'] == group).mean():.1%}")

station_clean
BLOOR STATION         1865
FINCH STATION         1816
KENNEDY BD STATION    1780
OTHER                 1779
KIPLING STATION       1734
UNION STATION         1158
YONGE BD STATION      1115
EGLINTON STATION      1105
VMC STATION           1097
WILSON STATION         932
Name: count, dtype: int64

YARD: 0.9%
LINE-WIDE: 1.3%
OTHER: 3.9%
UNKNOWN: 0.0%


In [17]:
# What ended up in OTHER?
df.loc[df["station_clean"] == "OTHER", "Station"].value_counts().head(25)

Station
YONGE-UNIVERSITY AND B    512
PIONEER VILLAGE STATIO    249
YORK UNIVERSITY STATIO    165
TORONTO TRANSIT COMMIS     78
YONGE UNIVERSITY AND B     25
NORTH YORK CENTRE STAT     21
YONGE/UNIVERSITY AND B     13
SCARBOROUGH CTR STATIO     12
UNION TO ST ANDREW STA     12
YONGE-UNIVERSITY-SPADI     11
YONGE UNIVERSITY SPADI      9
TRACK LEVEL ACTIVITY        9
UNION TO ST ANDREW          8
NORTH YORK CENTRE           8
DAVISVILLE BUILD UP         8
EARLY ACCESS CLOSURE -      8
OSSINGTON CENTRE TRACK      7
UNION TO KING               7
YUS                         7
SYSTEM WIDE                 7
DAVISVILLE BUILD-UP         6
VICTORIA PARK TO WARDE      5
SYSTEMWIDE                  5
MCBRIEN BUILDING            5
ST CLAIR WEST TO ST AN      5
Name: count, dtype: int64

In [18]:
# Cleaned station names that appear rarely: likely leftover variants
clean_counts = df["station_clean"].value_counts()
clean_counts[clean_counts < 50]

station_clean
ST GEORGE STATION         35
SPADINA YU STATION        19
UNION TO KING STATION     18
YONGE STATION             13
MAIN STATION               5
VARIOUS STATION            5
LEASIDE STATION            5
HWY 407 STATION            3
CEDARVALE STATION          3
SHPPARD STATION            2
KIPLING BD STATION         2
KING TO UNION STATION      2
AVENUE STATION             2
NORTH YORK STATION         2
SHEPPARD YONGE STATION     1
QUEEN QUAY STATION         1
WILSO STATION              1
QUEENS QUAY STATION        1
ASQUITH SUB STATION        1
JANE TO KEELE STATION      1
BAYVIEW SUB STATION        1
CELEDONIA STATION          1
OLD MILLS STATION          1
KENENDY STATION            1
WELLSELEY STATION          1
QUEENS PARK STATION        1
YORKDLAE STATION           1
GOLDEN MILE STATION        1
CALENDONIA STATION         1
DUNCAN SUB STATION         1
AVENUE ECLRT STATION       1
LANDSDOWNE STATION         1
KENNEDY ECLRT STATION      1
SHEPPARD SUB STATION       1
